In [ ]:
# Install required packages
!pip install earthengine-api requests pandas numpy -q

# Import libraries
import requests
import ee
import pandas as pd
import json
from datetime import datetime, timedelta
import numpy as np

print("📦 Packages installed successfully!")

# Earth Engine Authentication
print("🔐 Authenticating Earth Engine...")
try:
    ee.Authenticate()
    print("✅ Earth Engine authentication successful!")
except Exception as e:
    print("⚠️ Authentication may already be complete:", e)

# Initialize Earth Engine
try:
    ee.Initialize(project='modis-list')  # Use your project ID
    print("✅ Earth Engine initialized successfully!")
except Exception as e:
    print("❌ Earth Engine initialization failed:", e)
    ee.Initialize()
    print("✅ Earth Engine initialized!")

def generate_uttarakhand_grid():
    """Generate grid points covering entire Uttarakhand"""
    north = 31.5
    south = 28.5
    west = 77.5
    east = 81.0

    lats = np.arange(south, north, 0.1)
    lons = np.arange(west, east, 0.1)

    grid_points = []
    for lat in lats:
        for lon in lons:
            grid_points.append((lat, lon))

    print(f"📍 Generated {len(grid_points)} grid points")
    return grid_points

def fetch_openmeteo_batch(locations):
    """Fetch precipitation data for multiple locations in batch"""
    all_data = []
    today = datetime.now().strftime('%Y-%m-%d')

    # Process all locations
    print(f"🌧️ Processing {len(locations)} locations...")

    for i, (lat, lon) in enumerate(locations):
        try:
            url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=precipitation_sum,rain_sum,precipitation_probability_max,precipitation_hours&timezone=auto&forecast_days=7"

            response = requests.get(url, timeout=10)
            data = response.json()

            if 'daily' in data:
                feature = {
                    'latitude': lat,
                    'longitude': lon,
                    'date': data['daily']['time'][0],
                    'precipitation_sum': data['daily']['precipitation_sum'][0] or 0,
                    'rain_sum': data['daily']['rain_sum'][0] or 0,
                    'precipitation_probability_max': data['daily']['precipitation_probability_max'][0] or 0,
                    'precipitation_hours': data['daily']['precipitation_hours'][0] or 0,
                    'fetch_date': today,
                    'point_id': f"point_{i}"
                }
                all_data.append(feature)

            if (i + 1) % 10 == 0:
                print(f"✅ Processed {i+1}/{len(locations)} locations")

        except Exception as e:
            print(f"❌ Error for {lat}, {lon}: {e}")
            continue

    return all_data

def create_interpolation_image(data):
    """Create interpolated precipitation image"""
    if not data:
        print("❌ No data to interpolate!")
        return None

    df = pd.DataFrame(data)
    print(f"📊 Data summary - Min: {df['precipitation_sum'].min():.1f}mm, Max: {df['precipitation_sum'].max():.1f}mm, Mean: {df['precipitation_sum'].mean():.1f}mm")

    features = []
    for _, row in df.iterrows():
        geom = ee.Geometry.Point([row['longitude'], row['latitude']])
        feature = ee.Feature(geom, {
            'precipitation': row['precipitation_sum'],
            'rain': row['rain_sum'],
            'probability': row['precipitation_probability_max'],
            'hours': row['precipitation_hours']
        })
        features.append(feature)

    feature_collection = ee.FeatureCollection(features)

    precipitation_image = feature_collection.reduceToImage(
        ['precipitation'],
        ee.Reducer.mean()
    ).reproject('EPSG:4326', None, 5000)

    precipitation_smoothed = precipitation_image.focal_mean(5000, 'circle', 'meters')

    return precipitation_smoothed.rename('precipitation_sum')

def upload_daily_precipitation():
    """Main function to run daily"""
    print("🌧️ Starting Daily Uttarakhand Precipitation Update...")
    print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d')}")

    grid_points = generate_uttarakhand_grid()

    print("📡 Fetching precipitation data from Open-Meteo...")
    precipitation_data = fetch_openmeteo_batch(grid_points)

    if not precipitation_data:
        print("❌ No data fetched!")
        return None

    print(f"✅ Successfully fetched {len(precipitation_data)} data points")

    print("🔄 Creating interpolated precipitation image...")
    precipitation_image = create_interpolation_image(precipitation_data)

    if precipitation_image is None:
        print("❌ Failed to create precipitation image")
        return None

    today = datetime.now().strftime('%Y_%m_%d')

    try:
        # FIX: Create completely new asset path without training points folder
        asset_id = f'projects/modis-list/assets/PRECIPITATION_DAILY_UTTARAKHAND_{today}'

        print(f"☁️ Uploading to: {asset_id}")

        task = ee.batch.Export.image.toAsset(
            image=precipitation_image,
            description=f'Uttarakhand_Precipitation_{today}',
            assetId=asset_id,
            region=ee.Geometry.Rectangle([77.5, 28.5, 81.0, 31.5]),
            scale=5000,
            maxPixels=1e9
        )
        task.start()

        print(f"🚀 Upload started! Task ID: {task.id}")
        print("📊 Check task status at: https://code.earthengine.google.com/tasks")

        df = pd.DataFrame(precipitation_data)
        df.to_csv(f'uttarakhand_precipitation_{today}.csv', index=False)
        print("💾 Raw data saved to CSV")

        return asset_id

    except Exception as e:
        print(f"❌ Upload failed: {e}")
        return None

# Run the main function
print("=" * 50)
print("🚀 STARTING UTTARAKHAND PRECIPITATION PIPELINE")
print("=" * 50)

asset_id = upload_daily_precipitation()

if asset_id:
    print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"🔗 Asset ID: {asset_id}")
    print("📊 Check GEE Tasks: https://code.earthengine.google.com/tasks")
else:
    print("💥 PIPELINE FAILED!")

print("=" * 50)